# Tokenization Analysis Benchmark for Low-Resource Languages (Amharic vs. English)

**Author:** Bedru Yimam Ahmed 
**Email:** bedruy4@gmail.com

Evaluates tokenization inefficiencies, subword fragmentation, fertility ratios, and
context-window exhaustion across standard, multilingual, and Amharic-specialized
tokenizers, using 10 parallel Amharic/English sentence pairs.

Models covered:
- **Standard (Latin-centric):** Llama-3-8B, GPT-4o (`o200k_base` via `tiktoken`)
- **Multilingual baselines:** Multilingual-E5, XLM-RoBERTa-base
- **African/Amharic-focused:** Llama-3.2-1B-Amharic, LLAMA-Walia-II, RoBERTa-Amharic-Embed,
  Tiny Aya Earth, AfroXLMR-large


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from transformers import AutoTokenizer
import tiktoken
import matplotlib.pyplot as plt

## Helper functions: readable decoding (HuggingFace tokenizers)

In [ ]:
def decode_readable_tokens(tokenizer, string_tokens):
    """
    Convert tokens to human-readable form using the tokenizer's own decode
    logic, grouped into contiguous runs at word boundaries.

    IMPORTANT: this must decode tokens in GROUPS, not one-by-one. Some
    tokenizers (e.g. Llama-3's standard vocab on Amharic) fall back to
    single UTF-8 BYTES per token when a character has no dedicated merge.
    Since Ge'ez characters are 3 bytes in UTF-8, a lone byte token decoded
    in isolation is not valid UTF-8 by itself and will render as the
    replacement character '\ufffd'. Decoding a run of tokens together lets
    the bytes recombine into valid characters before decoding.

    Returns a list the SAME LENGTH as string_tokens, where each entry is
    the readable string for the run that token belongs to (so word-boundary
    tokens carry the full decoded chunk, and continuation tokens are shown
    as empty strings to avoid repeating the same text).
    """
    def is_word_start(tok):
        return tok.startswith("\u0120") or tok.startswith("\u2581") or tok == string_tokens[0]

    runs = []
    current_run = []
    for tok in string_tokens:
        if is_word_start(tok) and current_run:
            runs.append(current_run)
            current_run = [tok]
        else:
            current_run.append(tok)
    if current_run:
        runs.append(current_run)

    readable = []
    for run in runs:
        decoded_run = tokenizer.convert_tokens_to_string(run)
        readable.append(decoded_run)
        readable.extend([""] * (len(run) - 1))

    return readable


def decode_readable_string(tokenizer, string_tokens):
    """Convenience: full readable sentence (all runs decoded and joined)."""
    return tokenizer.convert_tokens_to_string(string_tokens)

## Helper functions: readable decoding (`tiktoken` — GPT-4o / GPT-4 / GPT-3.5)

`tiktoken` is not a HuggingFace tokenizer: it has no `convert_ids_to_tokens` /
`convert_tokens_to_string` API. Each token maps to raw **bytes**, not a string
with special markers like `\u0120`/`\u2581`. A word-start token's bytes begin
with a literal space byte (`b' '`), so we detect boundaries that way instead.
Decoding still has to happen in **groups** for the same reason as above:
Ge'ez characters are multi-byte in UTF-8, and a lone byte token is not valid
UTF-8 by itself.

In [ ]:
def tiktoken_readable_tokens_and_runs(encoding, token_ids):
    """
    tiktoken analogue of decode_readable_tokens: returns
    (raw_display_tokens, readable_tokens) both the same length as token_ids.
    raw_display_tokens: each token's own bytes decoded in isolation (may show
      the U+FFFD replacement character for split multi-byte characters --
      this is expected and mirrors the HF byte-fallback case).
    readable_tokens: each word-run decoded together (correct reconstruction),
      with continuation tokens shown as empty strings.
    """
    raw_display_tokens = [
        encoding.decode_single_token_bytes(t).decode("utf-8", errors="replace")
        for t in token_ids
    ]

    runs = []
    current_run = []
    for i, t in enumerate(token_ids):
        tok_bytes = encoding.decode_single_token_bytes(t)
        is_word_start = tok_bytes.startswith(b" ") or i == 0
        if is_word_start and current_run:
            runs.append(current_run)
            current_run = [t]
        else:
            current_run.append(t)
    if current_run:
        runs.append(current_run)

    readable = []
    for run in runs:
        decoded_run = encoding.decode(run)
        readable.append(decoded_run)
        readable.extend([""] * (len(run) - 1))

    return raw_display_tokens, readable

## Core analysis functions

In [ ]:
def analyze_tokenization_effects(text_samples, model_path, model_label, language_label):
    """Per-sentence tokenization metrics for one HuggingFace model, tagged by language."""
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    results = []
    for pair_id, text in enumerate(text_samples, start=1):
        words = text.split()
        word_count = len(words)
        char_count = len(text)

        tokens = tokenizer.encode(text, add_special_tokens=False)
        token_count = len(tokens)

        fertility = token_count / word_count if word_count > 0 else 0
        cpt = char_count / token_count if token_count > 0 else 0

        string_tokens = tokenizer.convert_ids_to_tokens(tokens)
        readable_tokens = decode_readable_tokens(tokenizer, string_tokens)
        readable_sentence = decode_readable_string(tokenizer, string_tokens)

        results.append({
            "Pair ID": pair_id,
            "Language": language_label,
            "Model/Tokenizer": model_label,
            "Text": text,
            "Word Count": word_count,
            "Token Count": token_count,
            "Fertility Ratio": fertility,
            "Chars Per Token (CPT)": cpt,
            "All Tokens": string_tokens,
            "All Tokens (Readable)": readable_tokens,
            "Readable Sentence": readable_sentence,
            "Fragmented Tokens Example": string_tokens[:8],
            "Fragmented Tokens Example (Readable)": readable_tokens[:8],
        })

    return pd.DataFrame(results)


def analyze_tokenization_effects_tiktoken(text_samples, encoding_name, model_label, language_label):
    """Per-sentence tokenization metrics for one tiktoken (GPT-family) model, tagged by language."""
    encoding = tiktoken.get_encoding(encoding_name)

    results = []
    for pair_id, text in enumerate(text_samples, start=1):
        words = text.split()
        word_count = len(words)
        char_count = len(text)

        token_ids = encoding.encode(text)
        token_count = len(token_ids)

        fertility = token_count / word_count if word_count > 0 else 0
        cpt = char_count / token_count if token_count > 0 else 0

        raw_display_tokens, readable_tokens = tiktoken_readable_tokens_and_runs(encoding, token_ids)
        readable_sentence = encoding.decode(token_ids)

        results.append({
            "Pair ID": pair_id,
            "Language": language_label,
            "Model/Tokenizer": model_label,
            "Text": text,
            "Word Count": word_count,
            "Token Count": token_count,
            "Fertility Ratio": fertility,
            "Chars Per Token (CPT)": cpt,
            "All Tokens": raw_display_tokens,
            "All Tokens (Readable)": readable_tokens,
            "Readable Sentence": readable_sentence,
            "Fragmented Tokens Example": raw_display_tokens[:8],
            "Fragmented Tokens Example (Readable)": readable_tokens[:8],
        })

    return pd.DataFrame(results)


def summarize_with_ci(df, model_label, language_label, confidence=0.95):
    """Aggregate per-sentence metrics into mean + CI per model, per language."""
    summary = {}
    for metric in ["Fertility Ratio", "Chars Per Token (CPT)"]:
        values = df[metric].values
        n = len(values)
        mean = np.mean(values)
        sem = stats.sem(values) if n > 1 else 0.0
        margin = sem * stats.t.ppf((1 + confidence) / 2, n - 1) if n > 1 else 0.0
        summary[f"{metric} (mean)"] = round(mean, 3)
        summary[f"{metric} (95% CI low)"] = round(mean - margin, 3)
        summary[f"{metric} (95% CI high)"] = round(mean + margin, 3)
    summary["Model/Tokenizer"] = model_label
    summary["Language"] = language_label
    summary["N sentences"] = len(df)
    return summary

## Parallel Amharic ↔ English sentence pairs

Each pair covers the same content/domain so cross-language fertility is
comparable (civic life, agriculture, health, education, infrastructure,
youth/entrepreneurship, vaccination). 


In [ ]:
parallel_pairs = [
    {
        "am": "በማህረሰባችን ውስጥ በተሳሳተ መረጃ ምክንያት የሚፈጠረውን ችግር መከላከል አለብን።",
        "en": "We must prevent the problems caused by misinformation in our community.",
    },
    {
        "am": "የኢትዮጵያ ግብርና ሚኒስቴር በድርቅ ክፍለ ጊዜ ገበሬዎችን ለመደገፍ እቅድ አውጥቷል።",
        "en": "Ethiopia's Ministry of Agriculture has developed a plan to support farmers during drought periods.",
    },
    {
        "am": "የመንገድ ትራፊክ አደጋዎችን ለመቀነስ አዲስ የደህንነት ደንብ ወጥቷል።",
        "en": "A new safety regulation has been issued to reduce road traffic accidents.",
    },
    {
        "am": "እናቶች በእርግዝና ወቅት መደበኛ የጤና ክትትል ማድረግ አለባቸው።",
        "en": "Mothers should receive regular health checkups during pregnancy.",
    },
    {
        "am": "የአየር ንብረት ለውጥ በኢትዮጵያ የግብርና ምርታማነት ላይ ተጽዕኖ እያሳደረ ነው።",
        "en": "Climate change is affecting agricultural productivity in Ethiopia.",
    },
    {
        "am": "ተማሪዎች በትምህርት ቤት ውስጥ ጥራት ያለው ትምህርት የማግኘት መብት አላቸው።",
        "en": "Students have the right to receive quality education in schools.",
    },
    {
        "am": "የከተማው አስተዳደር አዲስ የውሃ አቅርቦት ፕሮጀክት ጀምሯል።",
        "en": "The city administration has launched a new water supply project.",
    },
    {
        "am": "በአገሪቱ የተለያዩ ክፍሎች የኢንተርኔት አገልግሎት እየተስፋፋ ነው።",
        "en": "Internet service is expanding in various parts of the country.",
    },
    {
        "am": "ወጣቶች በስራ ፈጠራ ዘርፍ ስልጠና እንዲያገኙ ይበረታታሉ።",
        "en": "Young people are encouraged to receive training in entrepreneurship.",
    },
    {
        "am": "የጤና ባለሙያዎች ወቅታዊ ክትባት አስፈላጊነትን አጽንኦት ሰጥተዋል።",
        "en": "Health professionals have emphasized the importance of timely vaccination.",
    },
]

amharic_texts = [p["am"] for p in parallel_pairs]
english_texts = [p["en"] for p in parallel_pairs]

## Models under test

**HuggingFace models** (run through `AutoTokenizer`):
- Standard: Llama-3-8B
- Multilingual baselines: Multilingual-E5, **XLM-RoBERTa-base** *(new)*
- African/Amharic-specialized: Llama-3.2-1B-Amharic, LLAMA-Walia-II,
  RoBERTa-Amharic-Embed, Tiny Aya Earth, **AfroXLMR-large** *(new)*

**tiktoken model** (run through `tiktoken`, not `AutoTokenizer`):
- **GPT-4o / `o200k_base`** *(new)* — a commercial-API baseline. GPT tokenizer
  vocabularies are known to be even more Latin/English-centric than
  Llama-3's, so this is expected to be a strong (likely worst-case)
  comparison point for the "standard tokenizers fail on Amharic" claim.

In [ ]:
hf_models_to_test = [
    ("meta-llama/Meta-Llama-3-8B", "Llama-3-8B (Standard)"),
    ("intfloat/multilingual-e5-large", "Multilingual-E5 (Optimized)"),
    ("FacebookAI/xlm-roberta-base", "XLM-RoBERTa-base (Multilingual Baseline)"),
    ("rasyosef/Llama-3.2-1B-Amharic-Instruct", "Llama-3.2-1B-Amharic (Specialized)"),
    ("israel/LLAMA-Walia-II", "LLAMA-Walia-II (Specialized)"),
    ("rasyosef/roberta-amharic-text-embedding-base", "RoBERTa-Amharic-Embed (Specialized)"),
    ("CohereLabs/tiny-aya-earth", "Tiny Aya Earth (Multilingual/African-focused)"),
    ("Davlan/afro-xlmr-large", "AfroXLMR-large (African-focused)"),
]

tiktoken_models_to_test = [
    ("o200k_base", "GPT-4o (tiktoken o200k_base)"),
]

## Run the benchmark across all models and both languages

In [ ]:
per_sentence_results = []
summary_rows = []

for model_path, model_label in hf_models_to_test:
    df_am = analyze_tokenization_effects(amharic_texts, model_path, model_label, "Amharic")
    df_en = analyze_tokenization_effects(english_texts, model_path, model_label, "English")
    per_sentence_results.extend([df_am, df_en])
    summary_rows.append(summarize_with_ci(df_am, model_label, "Amharic"))
    summary_rows.append(summarize_with_ci(df_en, model_label, "English"))

for encoding_name, model_label in tiktoken_models_to_test:
    df_am = analyze_tokenization_effects_tiktoken(amharic_texts, encoding_name, model_label, "Amharic")
    df_en = analyze_tokenization_effects_tiktoken(english_texts, encoding_name, model_label, "English")
    per_sentence_results.extend([df_am, df_en])
    summary_rows.append(summarize_with_ci(df_am, model_label, "Amharic"))
    summary_rows.append(summarize_with_ci(df_en, model_label, "English"))

# Full per-sentence table, tagged by language and pair ID so Amharic/English
# rows for the same underlying sentence can be joined on (Model, Pair ID)
per_sentence_df = pd.concat(per_sentence_results, ignore_index=True)

# Aggregated summary table with 95% CIs, split by language -- this is the
# one for the paper
summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df[[
    "Model/Tokenizer", "Language", "N sentences",
    "Fertility Ratio (mean)", "Fertility Ratio (95% CI low)", "Fertility Ratio (95% CI high)",
    "Chars Per Token (CPT) (mean)", "Chars Per Token (CPT) (95% CI low)", "Chars Per Token (CPT) (95% CI high)",
]]

summary_df

## Amharic vs English "fertility gap" per model

How many more tokens/word each model needs for Amharic vs English on the
SAME underlying content. This is the headline number for the paper.

In [ ]:
pivot = summary_df.pivot(index="Model/Tokenizer", columns="Language", values="Fertility Ratio (mean)")
pivot["Fertility Gap (Amharic - English)"] = pivot["Amharic"] - pivot["English"]
pivot["Fertility Ratio (Amharic / English)"] = pivot["Amharic"] / pivot["English"]
pivot.round(3)

## Readable tokenization, side by side (Amharic vs English), first N pairs

Useful for pulling illustrative examples (e.g. worst fragmentation cases)
directly into the paper.

In [ ]:
N = 5
for pair_id in range(1, N + 1):
    pair_rows = per_sentence_df[per_sentence_df["Pair ID"] == pair_id]
    am_text = pair_rows[pair_rows["Language"] == "Amharic"]["Text"].iloc[0]
    en_text = pair_rows[pair_rows["Language"] == "English"]["Text"].iloc[0]
    print(f"Pair {pair_id}")
    print(f"  AM: {am_text}")
    print(f"  EN: {en_text}")
    print("-" * 80)
    for _, row in pair_rows.iterrows():
        print(f"  [{row['Language']}] {row['Model/Tokenizer']} "
              f"({row['Token Count']} tokens, fertility={row['Fertility Ratio']:.2f}):")
        print("    raw:      ", row["All Tokens"])
        print("    readable: ", row["All Tokens (Readable)"])
    print()

## Grouped bar chart: Amharic vs English fertility, per model, with 95% CI

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

models = summary_df["Model/Tokenizer"].unique()
x = np.arange(len(models))
width = 0.35

for i, lang in enumerate(["Amharic", "English"]):
    lang_df = summary_df[summary_df["Language"] == lang].set_index("Model/Tokenizer").loc[models]
    means = lang_df["Fertility Ratio (mean)"].values
    lower_err = means - lang_df["Fertility Ratio (95% CI low)"].values
    upper_err = lang_df["Fertility Ratio (95% CI high)"].values - means
    offset = (i - 0.5) * width
    ax.bar(x + offset, means, width, yerr=[lower_err, upper_err], capsize=4,
           label=lang, color="#4C72B0" if lang == "Amharic" else "#DD8452")

ax.set_ylabel("Fertility Ratio (tokens/word)")
ax.set_title("Amharic vs English Tokenization Fertility, Parallel Sentences (95% CI)")
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=25, ha="right")
ax.legend(title="Language")
plt.tight_layout()
plt.savefig("fertility_comparison_amharic_vs_english.png", dpi=150)
plt.show()